<a href="https://colab.research.google.com/github/simon-mellergaard/datavis/blob/main/Data/Indexing%20NLP%20algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Libraries needed**

In [1]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from huggingface_hub import notebook_login

notebook_login()

# **Load data**

In [3]:
import os
import sys
if 'google.colab' in sys.modules:
    %cd /content/
    # remove local directory if it already exists
    if os.path.isdir("datavis"):
        !rm -rf {"datavis"}
    !git clone https://github.com/simon-mellergaard/datavis.git
    %cd /content/datavis/Data

/content
Cloning into 'datavis'...
remote: Enumerating objects: 593, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 593 (delta 45), reused 50 (delta 20), pack-reused 506 (from 1)
Receiving objects: 100% (593/593), 40.47 MiB | 15.70 MiB/s, done.
Resolving deltas: 100% (335/335), done.
/content/datavis/Data


In [4]:
path = "DATA_UFM_combined.xlsx"
df = pd.read_excel(path)

# **Clustering**

In [11]:
import re
import os
import torch
from sentence_transformers import SentenceTransformer

COL_TITLE = "titel"
OUT_DIR = "/content/drive/MyDrive/Colab_Notebooks"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f"{OUT_DIR}/education_cluster_mapping.csv"
OUT_XLSX = f"{OUT_DIR}/education_cluster_mapping.xlsx"

df = df[df[COL_TITLE].astype(str).str.len() > 0].copy()

# Clusters with detailed subcategories (Undervisning removed)
clusters = [
    ("Byggeri - transport",
     "Uddannelser inden for byggeri, anlæg, konstruktion, arkitektur, byggeteknik. "
     "Håndværk som tømrer, murer, elektriker, VVS, maler. "
     "Transport og logistik, fragtmand, chauffør, speditør, lager."),

    ("Design - kunst",
     "Uddannelser inden for design, produktdesign, grafisk design, industrielt design, møbeldesign. "
     "Kunst, billedkunst, kunsthåndværk, keramik, glaskunst, maleri, skulptur."),

    ("Ernæring - sundhed - omsorg",
     "Mad og ernæring, kokkeuddannelse, ernæringsfaglig, gastronomi, fødevarer, bager, konditor. "
     "Sundhed og omsorg, sygeplejerske, læge, tandlæge, bioanalytiker, radiograf, jordemoder, fysioterapeut, ergoterapeut, social- og sundhedsassistent, SOSU. "
     "Træning og idræt, fitness, idrætslærer, personlig træner, kropsterapi."),

    ("Film - teater - musik",
     "Film og scenekunst, filmproduktion, skuespiller, teater, drama, sceneinstruktion, dramatiker. "
     "Musik og lyd, musikproduktion, komponist, musiker, lydtekniker, sangskriver."),

    ("Sikkerhed og forsvar",
     "Uddannelser inden for sikkerhed, forsvar, politi, militær, beredskab, brandvæsen, redningsberedskab, sikkerhedsvagt."),

    ("Handel - økonomi - markedsføring",
     "Forretning og handel, detailhandel, sælger, indkøb, salg, butiksassistent. "
     "Ledelse og forretningsudvikling, virksomhedsledelse, organisation, projektledelse, innovation, entrepreneurship. "
     "Markedsføring og kommunikation, marketing, reklame, PR, branding, digital markedsføring, kommunikationsrådgiver. "
     "Økonomi og finans, revisor, bogholder, finansrådgiver, bank, investering, regnskab, økonomi."),

    ("IT - teknik - produktion",
     "Industri og produktion, produktionsteknologi, fabrikation, industriel produktion, automation. "
     "IT og digital udvikling, programmering, softwareudvikling, webudvikling, datamatiker, datascience, IT-sikkerhed, netværk, systemadministration. "
     "Service og forsyning, energi, el-forsyning, varme, vand, kloak, vedligeholdelse. "
     "Teknik og mekanik, maskinteknik, mekanik, elektronik, automatik, robotteknologi, ingeniør."),

    ("Kultur - sprog - medier",
     "Kultur og turisme, kulturformidling, museumsvæsen, bibliotek, turismefaglig, rejseleder, eventkoordinator. "
     "Sprog og medier, journalist, forfatter, oversætter, tolk, kommunikation, medievidenskab, redaktør."),

    ("Natur - klima - miljø",
     "Dyr og fødevareproduktion, landmand, dyrepasser, veterinær, skovbrug, havebrug, gartneri, akvakultur, fiskeri. "
     "Klima og miljø, miljøteknologi, klimatilpasning, bæredygtighed, geografi, miljøplanlægning. "
     "Naturvidenskab og teknologi, biologi, kemi, fysik, bioteknologi, laborant, forskning."),

    ("Pædagogik - psykologi - sociale forhold",
     "Psykologi og sociale forhold, psykolog, socialrådgiver, socialpædagog, socialarbejder, misbrugsbehandler, familiebehandler. "
     "Pædagogik og læring, pædagog, børnehavepædagog, læring, udvikling, didaktik, undervisning, lærer, folkeskolelærer, gymnasielærer, erhvervsskolelærer, voksenunderviser, faglærer, underviser."),

    ("Samfund - kontor - forvaltning",
     "Kontor og administration, kontorassistent, sekretær, receptionist, HR, personaleadministration, sagsbehandling. "
     "Samfund og forvaltning, jura, advokat, offentlig forvaltning, politik, samfundsfag, statskundskab, international politik."),
]

cluster_labels = [c[0] for c in clusters]
label_texts = [f"{name}. {desc}" for name, desc in clusters]

def keyword_classify(title):
    """Pre-classify based on strong keyword matches"""
    title_lower = title.lower()

    # Define strong keywords for each cluster (Undervisning removed)
    keyword_rules = {
        "Ernæring - sundhed - omsorg": [
            r'\bidræt', r'\bsport', r'\bfitness', r'\btræn',
            r'\bsygepl', r'\bsundhed', r'\bomsorg', r'\bsosu\b',
            r'\bkok\b', r'\bernæring', r'\bgastronom',
            r'\bfysiotera', r'\bergotera', r'\bjordemoder',
            r'\blæge\b', r'\btandlæge', r'\bbioanalyt', r'\bradiograf',
            r'\bsocial.*sundhed', r'\bplej', r'\bterapi\b',
            r'\bbager\b', r'\bkonditor'
        ],
        "Handel - økonomi - markedsføring": [
            r'\bøkonom', r'\bfinans', r'\brevis', r'\bboghol',
            r'\bhandel', r'\bmarket', r'\bsalg', r'\bindkøb',
            r'\bHA\b', r'\bHD\b', r'\bcand\.merc', r'\bmerc\b',
            r'\bforretning', r'\bkommerciel', r'\bdetailhandel',
            r'\bledelse\b', r'\binnovation\b'
        ],
        "IT - teknik - produktion": [
            r'\bdatamat', r'\bIT\b', r'\bprogramm', r'\bsoftware',
            r'\bweb', r'\bdata science', r'\bingeniør', r'\bteknolog',
            r'\bautomation', r'\bproduktion', r'\bmekanik', r'\belektronik',
            r'\brobottek', r'\bdigital udvikling', r'\bnetværk'
        ],
        "Kultur - sprog - medier": [
            r'\bjournalist', r'\bmedie', r'\boversæt', r'\btolk\b',
            r'\bsprog', r'\bkommunikation', r'\bkulturformid',
            r'\bbibliotek', r'\bturisme', r'\bredaktør', r'\bmuseum'
        ],
        "Pædagogik - psykologi - sociale forhold": [
            r'\bpsykolog', r'\bsocial.*rådgiv', r'\bsocialpæd',
            r'\bsocialfag', r'\bpædagog', r'\bbørnehave.*pæd',
            r'\bmisbrugsbehandl', r'\bfamiliebehandl',
            r'\blærer', r'\bunderviser', r'\bfolkeskole.*lærer',
            r'\bgymnas.*lærer', r'\berhvervsskole.*lærer',
            r'\bvoksenunderviser', r'\bfaglærer', r'\bdidaktik'
        ],
        "Design - kunst": [
            r'\bdesign', r'\bkunst', r'\bgrafisk', r'\bindustrielt design',
            r'\bmøbeldesign', r'\bproduktdesign', r'\bkeramik',
            r'\bglaskunst', r'\bmaleri\b', r'\bskulptur'
        ],
        "Film - teater - musik": [
            r'\bfilm', r'\bteater', r'\bmusik', r'\bskuespil',
            r'\bdrama\b', r'\bscene', r'\bkomponist', r'\bmusikproduktion',
            r'\blydtekniker', r'\bsangskriver'
        ],
        "Natur - klima - miljø": [
            r'\bbiolog', r'\bkemi\b', r'\bfysik\b', r'\bmiljø',
            r'\bklima', r'\blandmand', r'\bskovbrug', r'\blaborant',
            r'\bdyrepasser', r'\bveterinær', r'\bhavebrug', r'\bgartn',
            r'\bakvakultur', r'\bfiskeri', r'\bbæredygtig', r'\bgeograf'
        ],
        "Samfund - kontor - forvaltning": [
            r'\bjura\b', r'\badvokat', r'\bforvaltning', r'\bkontor',
            r'\bstatskundskab', r'\bsamfundsfag', r'\bcand\.jur',
            r'\bsekretær', r'\breceptionist', r'\bHR\b',
            r'\bpersonaleadmin', r'\bsagsbehandl', r'\bpolitik'
        ],
        "Byggeri - transport": [
            r'\bbygge', r'\bmurer\b', r'\btømrer', r'\belektriker',
            r'\bVVS\b', r'\btransport', r'\blogistik', r'\bchauffør',
            r'\banlæg', r'\bkonstruktion', r'\barkitektur',
            r'\bmaler\b', r'\bspeditør', r'\bhåndværk'
        ],
        "Sikkerhed og forsvar": [
            r'\bpoliti', r'\bmilitær', r'\bforsvar', r'\bbrand',
            r'\bberedskab', r'\bsikkerhed.*vagt', r'\brednings'
        ],
    }

    # Check each cluster's keywords
    for cluster, patterns in keyword_rules.items():
        for pattern in patterns:
            if re.search(pattern, title_lower):
                return cluster, 0.95  # High confidence score

    return None, 0.0  # No match

# Initialize columns
df["cluster_label"] = None
df["cluster_score"] = 0.0

# First pass: keyword-based classification
print("Pass 1: Keyword-based classification...")
keyword_matches = 0
for idx, row in df.iterrows():
    title = row[COL_TITLE]
    cluster, score = keyword_classify(title)
    if cluster:
        df.at[idx, "cluster_label"] = cluster
        df.at[idx, "cluster_score"] = score
        keyword_matches += 1

print(f"  Classified {keyword_matches} programs via keywords")

# Second pass: semantic matching for unclassified items
unclassified_mask = df["cluster_label"].isna()
unclassified_count = unclassified_mask.sum()

if unclassified_count > 0:
    print(f"Pass 2: Semantic matching for {unclassified_count} remaining programs...")

    # Danish state-of-the-art model
    MODEL_NAME = "KennethEnevoldsen/dfm-sentence-encoder-large-1"  # Danish Foundation Models sentence encoder
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"  Loading Danish model: {MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME, device=device)

    # Encode cluster descriptions
    label_emb = model.encode(
        label_texts,
        batch_size=32,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    # Encode unclassified titles
    unclassified_titles = df.loc[unclassified_mask, COL_TITLE].astype(str).str.strip().add(" uddannelse").tolist()

    text_emb = model.encode(
        unclassified_titles,
        batch_size=128,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    sims = text_emb @ label_emb.T
    best_scores, best_idx = torch.max(sims, dim=1)

    df.loc[unclassified_mask, "cluster_label"] = [cluster_labels[i] for i in best_idx.tolist()]
    df.loc[unclassified_mask, "cluster_score"] = best_scores.detach().cpu().numpy()

# Low-confidence bucket
THRESHOLD = 0.20
low_conf_count = (df["cluster_score"] < THRESHOLD).sum()
df.loc[df["cluster_score"] < THRESHOLD, "cluster_label"] = "Øvrige/Ukendt"
print(f"Moved {low_conf_count} low-confidence programs to 'Øvrige/Ukendt'")

# Save to CSV and Excel
out = df[[COL_TITLE, "cluster_label", "cluster_score"]].copy()
out.to_csv(OUT_CSV, index=False)
out.to_excel(OUT_XLSX, index=False)

print(f"\nSaved CSV:   {OUT_CSV}")
print(f"Saved Excel: {OUT_XLSX}")
print("\nCounts per cluster:")
print(out["cluster_label"].value_counts())

Pass 1: Keyword-based classification...
  Classified 982 programs via keywords
Pass 2: Semantic matching for 1167 remaining programs...
  Loading Danish model: KennethEnevoldsen/dfm-sentence-encoder-large-1


config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Some weights of the model checkpoint at KennethEnevoldsen/dfm-sentence-encoder-large-1 were not used when initializing BertModel: ['lm_head.bias', 'lm_head.decoder.weight', 'lm_head.transform.LayerNorm.bias', 'lm_head.transform.LayerNorm.weight', 'lm_head.transform.dense.bias', 'lm_head.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/379 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Moved 0 low-confidence programs to 'Øvrige/Ukendt'

Saved CSV:   /content/drive/MyDrive/Colab_Notebooks/education_cluster_mapping_2.csv
Saved Excel: /content/drive/MyDrive/Colab_Notebooks/education_cluster_mapping_2.xlsx

Counts per cluster:
cluster_label
IT - teknik - produktion                   456
Samfund - kontor - forvaltning             262
Handel - økonomi - markedsføring           250
Natur - klima - miljø                      211
Ernæring - sundhed - omsorg                211
Byggeri - transport                        198
Pædagogik - psykologi - sociale forhold    161
Kultur - sprog - medier                    145
Design - kunst                             130
Film - teater - musik                       82
Sikkerhed og forsvar                        43
Name: count, dtype: int64
